# 3. Riesgo de cobro

**Pregunta:** ¿qué facturas a crédito se van a pagar tarde y a qué clientes conviene revisar el crédito?

**Una fila = una factura (o boleta) a crédito.** **Objetivo:** `pago_tardio` = 1 si se cobró con **más de 15 días de retraso** sobre el vencimiento, o si a la fecha de cierre lleva más de 15 días vencida sin pagar.

**Método:**
1. Para cada factura se calcula **el historial de pago del cliente hasta el día de su emisión** (cuántas facturas pagó tarde antes, retraso medio, facturas vencidas pendientes). Es la variable más informativa y hay que construirla sin mirar al futuro.
2. Partición temporal: se entrena con facturas emitidas hasta septiembre de 2025 y se prueba con las posteriores.
3. Se compara un *gradient boosting*, una regresión logística y una regla simple ("% de facturas tardías del cliente").
4. Se aplica a la cartera pendiente y a cada cliente.

**Métricas:** AUC (ver cuaderno 02) y **precisión media** (*average precision*): qué tan concentradas quedan las facturas tardías arriba de la lista. Es más exigente que el AUC cuando la clase positiva es minoritaria (aquí, el 15 %).

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

from comun import *   # conexion al almacen (con, q), CORTE, guardar(), estilo(), mape(), colores

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

UMBRAL = 15  # días de retraso a partir de los cuales el pago se considera tardío

## 1. Datos: una fila por factura a crédito

In [ ]:
docs = q('''
    SELECT v.serie || '-' || v.numero AS documento, ANY_VALUE(c.cliente_id) AS cliente_id,
           ANY_VALUE(c.nombre) AS cliente, ANY_VALUE(c.tipo_cliente) AS tipo_cliente,
           ANY_VALUE(c.agente_retencion) AS agente_retencion, ANY_VALUE(cp.dias_credito) AS dias_credito,
           ANY_VALUE(m.codigo_iso) AS moneda, CAST(ANY_VALUE(fe.fecha) AS DATE) AS f_emision,
           CAST(ANY_VALUE(fv.fecha) AS DATE) AS f_vencimiento, SUM(v.importe_neto_pen + v.igv_pen) AS total_pen,
           COUNT(DISTINCT p.familia) AS n_familias,
           MAX(CASE WHEN p.linea_negocio <> 'VENTA DE MATERIALES' THEN 1 ELSE 0 END) AS incluye_servicio
    FROM fact_ventas v
    JOIN dim_tipo_documento t  ON t.tipo_documento_key = v.tipo_documento_key
    JOIN dim_condicion_pago cp ON cp.condicion_pago_key = v.condicion_pago_key
    JOIN dim_cliente c         ON c.cliente_key = v.cliente_key
    JOIN dim_moneda m          ON m.moneda_key = v.moneda_key
    JOIN dim_producto p        ON p.producto_key = v.producto_key
    JOIN dim_fecha fe          ON fe.fecha_key = v.fecha_emision_key
    JOIN dim_fecha fv          ON fv.fecha_key = v.fecha_vencimiento_key
    WHERE t.signo = 1 AND cp.es_credito
    GROUP BY 1
''')
cartera = q('SELECT documento, saldo_pen, dias_vencido, tramo_antiguedad FROM v_cartera_pendiente')
ultimo_cobro = q('''
    SELECT serie_doc || '-' || numero_doc AS documento, MAX(CAST(f.fecha AS DATE)) AS f_ultimo_cobro
    FROM fact_cobros c JOIN dim_fecha f ON f.fecha_key = c.fecha_cobro_key
    WHERE tipo_movimiento = 'COBRO' GROUP BY 1
''')
d = docs.merge(cartera, on='documento', how='left').merge(ultimo_cobro, on='documento', how='left')
for c in ('f_emision', 'f_vencimiento', 'f_ultimo_cobro'):
    d[c] = pd.to_datetime(d[c])
print(len(d), 'facturas a crédito')

## 2. La variable objetivo

- **Pagada** (no está en la cartera pendiente): retraso = fecha del último cobro − vencimiento.
- **Pendiente**: retraso = fecha de cierre − vencimiento (lo que lleva vencida).

Una factura emitida hace poco aún no tiene resultado conocido (no sabemos si se pagará a tiempo). Solo se usan para entrenar las facturas **etiquetables**: las pagadas a tiempo, o las que ya han pasado el vencimiento + 15 días.

In [ ]:
d['pagado'] = d.saldo_pen.isna()
d['retraso_dias'] = np.where(d.pagado, (d.f_ultimo_cobro - d.f_vencimiento).dt.days, (CORTE - d.f_vencimiento).dt.days)
d['pago_tardio'] = (d.retraso_dias > UMBRAL).astype(int)
# Día en que se conoce el resultado: el del pago si fue a tiempo; si no, vencimiento + 16 días
d['f_resultado'] = pd.to_datetime(np.where(d.pagado & (d.retraso_dias <= UMBRAL), d.f_ultimo_cobro,
                                           d.f_vencimiento + pd.Timedelta(days=UMBRAL + 1)))
d['etiquetable'] = d.f_resultado <= CORTE
d = d.sort_values('f_emision').reset_index(drop=True)
print(f"Etiquetables: {d.etiquetable.sum()}  |  pagos tardíos: {d[d.etiquetable].pago_tardio.mean():.1%}")

## 3. Historial del cliente sin mirar al futuro

Para cada factura se mira **solo lo que ya se sabía el día de su emisión**: las facturas anteriores del mismo cliente cuyo resultado ya se conocía. Esta es la parte más delicada del cuaderno. Si se usara el historial completo, el modelo aprendería del futuro y en la prueba parecería mejor de lo que es.

In [ ]:
def historial(cliente_id, fecha):
    previas = d[(d.cliente_id == cliente_id) & (d.f_emision < fecha)]
    conocidas = previas[previas.f_resultado < fecha]
    vencidas_sin_pagar = previas[(previas.f_vencimiento < fecha) & ~(previas.pagado & (previas.f_ultimo_cobro < fecha))]
    return {'facturas_previas': len(conocidas),
            'pct_tardias_previas': conocidas.pago_tardio.mean() if len(conocidas) else -1.0,   # -1 = sin historial
            'retraso_medio_previo': conocidas.retraso_dias.clip(-30, 120).mean() if len(conocidas) else 0.0,
            'vencidas_sin_pagar': len(vencidas_sin_pagar)}


d = pd.concat([d, pd.DataFrame([historial(r.cliente_id, r.f_emision) for r in d.itertuples()])], axis=1)
d['sin_historial'] = (d.facturas_previas == 0).astype(int)
d['log_total'] = np.log1p(d.total_pen)
d[['documento', 'cliente', 'f_emision', 'facturas_previas', 'pct_tardias_previas', 'retraso_medio_previo', 'pago_tardio']].tail()

## 4. Entrenamiento y evaluación

In [ ]:
NUMERICAS = ['log_total', 'dias_credito', 'n_familias', 'incluye_servicio', 'facturas_previas',
             'pct_tardias_previas', 'retraso_medio_previo', 'vencidas_sin_pagar', 'sin_historial']
CATEGORICAS = ['tipo_cliente', 'moneda', 'agente_retencion']
NOMBRES = {'log_total': 'Importe de la factura', 'dias_credito': 'Días de crédito', 'n_familias': 'Familias en la factura',
           'incluye_servicio': 'Incluye servicio o alquiler', 'facturas_previas': 'Facturas previas del cliente',
           'pct_tardias_previas': '% de facturas pagadas tarde antes', 'retraso_medio_previo': 'Retraso medio anterior',
           'vencidas_sin_pagar': 'Facturas vencidas sin pagar', 'sin_historial': 'Cliente sin historial',
           'tipo_cliente': 'Tipo de cliente', 'moneda': 'Moneda', 'agente_retencion': 'Agente de retención'}
X = NUMERICAS + CATEGORICAS


def nuevo_modelo(tipo):
    pre = ColumnTransformer([('num', StandardScaler(), NUMERICAS),
                             ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAS)], sparse_threshold=0)
    if tipo == 'gb':
        return make_pipeline(pre, HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=250,
                                                                 random_state=SEMILLA))
    return make_pipeline(pre, LogisticRegression(max_iter=3000, C=0.5))


etiq = d[d.etiquetable]
entren, prueba = etiq[etiq.f_emision < '2025-10-01'], etiq[etiq.f_emision >= '2025-10-01']
gb = nuevo_modelo('gb').fit(entren[X], entren.pago_tardio)
lr = nuevo_modelo('lr').fit(entren[X], entren.pago_tardio)
p_gb = gb.predict_proba(prueba[X])[:, 1]
p_lr = lr.predict_proba(prueba[X])[:, 1]
p_regla = prueba.pct_tardias_previas.where(prueba.pct_tardias_previas >= 0, entren.pago_tardio.mean())

metricas = pd.DataFrame([
    {'modelo': nombre, 'auc': roc_auc_score(prueba.pago_tardio, p), 'precision_media': average_precision_score(prueba.pago_tardio, p),
     'brier': brier_score_loss(prueba.pago_tardio, p)}
    for nombre, p in [('Gradient boosting', p_gb), ('Regresion logistica', p_lr),
                      ('Regla simple: % de facturas tardias del cliente', p_regla)]]).round(3)
metricas = metricas.assign(n_entrenamiento=len(entren), n_prueba=len(prueba),
                           tasa_tardias_prueba_pct=round(prueba.pago_tardio.mean() * 100, 1))
guardar(metricas, 'riesgo_cobro_metricas')
metricas

In [ ]:
marca = p_gb >= 0.5
print(f'Con umbral 0,5, el modelo marca {marca.sum()} facturas de {len(prueba)}:')
print(f'  precisión:     {prueba.pago_tardio[marca].mean():.0%} de las marcadas se pagaron tarde')
print(f'  exhaustividad: detecta el {marca[prueba.pago_tardio.values == 1].mean():.0%} de las facturas tardías')

**Cómo leerlo:** el AUC es muy alto (≈ 0,98) y el modelo mejora a la regla simple sobre todo en precisión media. La razón es que **cada cliente paga de forma muy constante**: quien pagó tarde antes vuelve a pagar tarde.

⚠️ **Matiz para la memoria:** un AUC tan alto se explica porque, en el histórico analizado, cada cliente paga de forma muy estable. Si los hábitos de pago cambiaran (una crisis del sector, clientes nuevos sin historial), el rendimiento bajaría. Conviene volver a medirlo cada trimestre con los datos nuevos.

## 5. ¿Qué variables pesan más? (importancia por permutación)

Se desordena una variable cada vez y se mide cuánto cae el AUC. Si cae mucho, el modelo depende de ella.

In [ ]:
imp = permutation_importance(gb, prueba[X], prueba.pago_tardio, scoring='roc_auc', n_repeats=20, random_state=SEMILLA)
importancia = pd.DataFrame({'variable': [NOMBRES[v] for v in X], 'perdida_auc_si_se_quita': imp.importances_mean.round(4)}) \
    .sort_values('perdida_auc_si_se_quita', ascending=False)
guardar(importancia, 'riesgo_cobro_importancia')
importancia

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10.5, 4))
for p, nombre, color in [(p_gb, 'Gradient boosting', AZUL), (p_regla, 'Regla simple', GRIS)]:
    fpr, tpr, _ = roc_curve(prueba.pago_tardio, p)
    axs[0].plot(fpr, tpr, color=color, lw=2, label=f'{nombre} (AUC {roc_auc_score(prueba.pago_tardio, p):.2f})')
axs[0].plot([0, 1], [0, 1], color='#ccc', ls='--', lw=1)
axs[0].set_xlabel('Facturas puntuales marcadas por error'); axs[0].set_ylabel('Facturas tardías detectadas')
estilo(axs[0], 'Curva ROC en el periodo de prueba'); axs[0].legend(frameon=False, fontsize=8, loc='lower right')
top = importancia.head(6).iloc[::-1]
axs[1].barh(top.variable, top.perdida_auc_si_se_quita, color=AZUL)
estilo(axs[1], 'Qué variables pesan más'); axs[1].set_xlabel('Pérdida de AUC si se desordena la variable')
axs[1].grid(axis='x', color='#e5e5e5'); axs[1].grid(axis='y', visible=False)
guardar_figura(fig, '3_riesgo_cobro')
plt.show()

## 6. Aplicación: cartera pendiente y clientes

Se reentrena con todas las facturas etiquetables y se aplica a:
- **la cartera pendiente**: probabilidad de que cada factura aún no vencida se pague tarde;
- **cada cliente**: probabilidad de que su próxima factura a crédito se pague tarde, con lo que se sabe al cierre.

In [ ]:
gb = nuevo_modelo('gb').fit(etiq[X], etiq.pago_tardio)

pend = d[~d.pagado].copy()
pend['prob_pago_tardio'] = gb.predict_proba(pend[X])[:, 1].round(3)
pend['situacion'] = np.select(
    [pend.retraso_dias > UMBRAL, pend.prob_pago_tardio >= 0.5, pend.prob_pago_tardio >= 0.2],
    ['YA EN MORA (vencida hace mas de 15 dias)', 'RIESGO ALTO', 'RIESGO MEDIO'], 'RIESGO BAJO')
pend.loc[pend.retraso_dias > UMBRAL, 'prob_pago_tardio'] = np.nan  # ya se sabe: no hace falta predecir
pend['f_emision'] = pend.f_emision.dt.date
pend['f_vencimiento'] = pend.f_vencimiento.dt.date
cartera_riesgo = pend[['documento', 'cliente_id', 'cliente', 'tipo_cliente', 'f_emision', 'f_vencimiento', 'saldo_pen',
                       'dias_vencido', 'tramo_antiguedad', 'prob_pago_tardio', 'situacion']] \
    .sort_values(['situacion', 'saldo_pen'], ascending=[True, False])
guardar(cartera_riesgo, 'cartera_pendiente_riesgo')
cartera_riesgo.groupby('situacion').agg(facturas=('documento', 'size'), saldo_pen=('saldo_pen', 'sum')).round(0)

In [ ]:
hace_un_anio = CORTE - pd.Timedelta(days=365)
cli = d.groupby('cliente_id').agg(
    cliente=('cliente', 'first'), tipo_cliente=('tipo_cliente', 'first'), agente_retencion=('agente_retencion', 'last'),
    moneda=('moneda', 'last'), dias_credito=('dias_credito', 'last'), total_pen=('total_pen', 'median'),
    n_familias=('n_familias', 'median'), incluye_servicio=('incluye_servicio', 'max'),
    facturas_credito=('documento', 'size')).reset_index()
cli['importe_credito_12m_pen'] = cli.cliente_id.map(d[d.f_emision > hace_un_anio].groupby('cliente_id').total_pen.sum()).fillna(0).round(2)
# Una factura "típica" del cliente, emitida el día después del cierre
cli = pd.concat([cli, pd.DataFrame([historial(c, CORTE + pd.Timedelta(days=1)) for c in cli.cliente_id])], axis=1)
cli['sin_historial'] = (cli.facturas_previas == 0).astype(int)
cli['log_total'] = np.log1p(cli.total_pen)
cli['prob_pago_tardio_proxima'] = gb.predict_proba(cli[X])[:, 1].round(3)
saldo = cartera.merge(d[['documento', 'cliente_id']], on='documento').groupby('cliente_id').saldo_pen.sum()
cli['saldo_pendiente_pen'] = cli.cliente_id.map(saldo).fillna(0).round(2)
cli['recomendacion'] = np.select([cli.prob_pago_tardio_proxima >= 0.5, cli.prob_pago_tardio_proxima >= 0.2],
                                 ['REVISAR CREDITO: pedir adelanto o reducir plazo',
                                  'VIGILAR: recordatorio antes del vencimiento'], 'SIN ACCION')
cli['pct_tardias_previas'] = (cli.pct_tardias_previas.where(cli.pct_tardias_previas >= 0) * 100).round(1)
cli['retraso_medio_previo'] = cli.retraso_medio_previo.round(1)
clientes_riesgo = cli[['cliente_id', 'cliente', 'tipo_cliente', 'facturas_credito', 'facturas_previas', 'pct_tardias_previas',
                       'retraso_medio_previo', 'vencidas_sin_pagar', 'saldo_pendiente_pen', 'importe_credito_12m_pen',
                       'prob_pago_tardio_proxima', 'recomendacion']].sort_values('prob_pago_tardio_proxima', ascending=False)
guardar(clientes_riesgo, 'clientes_riesgo_cobro')
clientes_riesgo.recomendacion.value_counts()

## Conclusiones (para la memoria)

- En torno al 15 % de las facturas a crédito se paga con más de 15 días de retraso.
- El mejor predictor, con diferencia, es **cómo pagó el cliente antes**. El importe, el plazo o la familia de producto apenas influyen.
- El modelo permite pasar de una gestión de cobro reactiva (reclamar cuando ya está vencida) a una **preventiva**: revisar el crédito de los clientes marcados antes de venderles a plazo.
